In [13]:
import os
import json
import numpy as np
import random
import time
import re
import tiktoken
from google import genai
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from openai import OpenAI

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [12]:

# --- Configuration ---
EMBEDDING_DIM = 384  # Dimension for question and answer embeddings

UPDATE_FREQUENCY = 1    # Update JSON records after every question


# TODO: configure more parameters here
# Regularization parameter λ, exploration coefficient γ, step size η, network width m, 
# gradient descent steps J, LLM pool size M , batch size B_batch

LEARNING_RATE = 0.01 # 学习率
REG = 1 #正则化参数
M = 7 # LLM pool size
GEMMA = 1 # 探索系数
L = 3 # 2层隐藏层
BATCH_SIZE = 10 # B_batch
WIDTH = 100 # m
GD_STEPS = 10 # J

In [4]:
# --- CUDA / Device Setup ---
# 自动选择 GPU；若不可用则回退到 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA runtime: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("Using CPU")


def to_device(x):
    """Move tensor/module to selected device."""
    if hasattr(x, "to"):
        return x.to(DEVICE)
    return x


# 可选：提高矩阵乘法性能（Ampere 及以上通常有效）
torch.set_float32_matmul_precision("high")

Torch version: 2.10.0+cu128
CUDA available: True
CUDA runtime: 12.8
GPU count: 2
Current GPU: Quadro RTX 8000


In [5]:
if DEVICE.type == "cuda":
    # 清理缓存（可选）
    torch.cuda.empty_cache()
    print("CUDA is ready for training/inference.")

CUDA is ready for training/inference.


In [6]:
from dotenv import load_dotenv, find_dotenv
from pathlib import Path

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    # Notebook 常在 Algorithms/ 目录运行，.env 在项目根目录
    candidates = [Path.cwd() / ".env", Path.cwd().parent / ".env"]
    for p in candidates:
        if p.exists():
            dotenv_path = str(p)
            break
if dotenv_path:
    load_dotenv(dotenv_path, override=False)
    print("Loaded .env:", dotenv_path)
else:
    print("No .env found; please create one at project root.")


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")

Loaded .env: /home/guisy/MyExperiment/LLM_DAG_Routing_02/.env


In [9]:
# --- Test two models via OpenRouter ---
Decompose_MODEL_NAME = "deepseek/deepseek-v3.2"
GRADER_MODEL_NAME = "deepseek/deepseek-v3.2"

assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY 未设置，请检查 .env"
base_url = OPENROUTER_BASE_URL 

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=base_url)

models_to_test = [Decompose_MODEL_NAME, GRADER_MODEL_NAME]
prompt = "请只回复：OK"

for model_name in models_to_test:
    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=16,
            temperature=0.0,
        )
        text = resp.choices[0].message.content if resp.choices else "<no choice>"
        print(f"✅ {model_name} -> {text}")
    except Exception as e:
        print(f"❌ {model_name} failed: {type(e).__name__}: {e}")

✅ deepseek/deepseek-v3.2 -> OK
✅ deepseek/deepseek-v3.2 -> OK


In [10]:
# 从项目根目录的 model_config.py 加载模型列表
import sys
from pathlib import Path

# 当前 notebook 在 Algorithm/ 下，根目录是其上一级
project_root = Path.cwd() if (Path.cwd() / "model_config.py").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from model_config import MODELS_CONFIG

AVAILABLE_LLMS = list(MODELS_CONFIG.keys())
LLM_ID_DIM = len(AVAILABLE_LLMS)

print(f"Loaded {LLM_ID_DIM} models from model_config.py")
print(AVAILABLE_LLMS[:5])

Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct']


In [ ]:
# TODO: load train set data/fused_qa_500.json
# TODO: 创建记录模型回答的json文件，记录格式为：
# {
#   "problem_id": {
#       "question": "原问题文本",
#       "answers": "原问题答案",
#       "final_status": "success/failure",
#       "steps_taken": 3, #几个节点
#       "attempts": [
#        {
#         "step": ,
#         "task_desc": "",
#         "chosen_model": "",
#         "reward": ,
#         "output": "",
#         "llm_input_messages": [
#           {
#             "role": "system",
#             "content": ""
#           },
#           {
#             "role": "user",
#             "content": ""
#           }
#         ]
#        },
#        { 
#         "step": "2",
#         "task_desc": "",
#         "chosen_model": "",
#         "reward": ,
#         "output": "",
#         "llm_input_messages": [
#           {
#             "role": "system",
#             "content": ""
#           },
#           {
#             "role": "user",
#             "content": ""
#           }
#         ]
#        }
#       ]
#
#   }
# }

# TODO: 保存模型参数文件
# TODO: complete .gitignore to exclude model parameters

In [ ]:
EMBEDDING_MODEL = ""
# TODO: choose an embedding model that can process long contexts.

In [ ]:
# TODO
# 将上下文拼接好后再转换为384维向量
class FeatureExtractor:

In [ ]:
# TODO: test Decompose_MODEL_NAME、GRADER_MODEL_NAME ability to decompose and grade.

In [ ]:
# TODO: ReLU需要改吗

In [ ]:
class LLMNet(nn.Module):
    def __init__(self, input_dim=EMBEDDING_DIM, width=WIDTH, L=L):
        super().__init__()
        num_hidden = max(L - 1, 1)
        layers = []
        in_dim = input_dim
        for _ in range(num_hidden):
            layers.append(nn.Linear(in_dim, width))
            layers.append(nn.ReLU())
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if not torch.is_tensor(x):
            x = torch.tensor(x, dtype=torch.float32)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        return self.net(x).squeeze(-1)


In [ ]:
class DAGNode:
    def __init__(self, node_id, task_description):
        self.node_id = node_id
        self.task_description = task_description
        self.predecessors = []
        self.successors = []
        self.layer = 0
        
        # 运行时状态
        self.input_context = ""
        self.output_content = ""
        self.selected_model = None
        self.feature_vector = None 
        self.reward = 0.0          

    def add_successor(self, node):
        if node not in self.successors:
            self.successors.append(node)
        if self not in node.predecessors:
            node.predecessors.append(self)

class DAGGraph:
    def __init__(self):
        self.nodes = {}

    def add_node(self, node_id, description):
        if node_id not in self.nodes:
            self.nodes[node_id] = DAGNode(node_id, description)
        return self.nodes[node_id]

    def add_edge(self, source_id, target_id):
        if source_id in self.nodes and target_id in self.nodes:
            self.nodes[source_id].add_successor(self.nodes[target_id])

    def compute_layers(self):
        #层级计算：Layer = max(predecessor_layers) + 1
        for node in self.nodes.values(): node.layer = 0
        for _ in range(len(self.nodes) + 1):
            for node in self.nodes.values():
                for pred in node.predecessors:
                    if node.layer < pred.layer + 1:
                        node.layer = pred.layer + 1
        layers = {}
        for node in self.nodes.values():
            if node.layer not in layers: layers[node.layer] = []
            layers[node.layer].append(node)
        return layers


In [ ]:
# TODO: 实现 GreedyNeuralUCBModel 类，包含参数初始化、更新、预测等方法。（抄过来还没修改完）
class GreedyNeuralUCBModel:
    def __init__(
        self,
        model_names,
        feature_dim=EMBEDDING_DIM,
        learning_rate=LEARNING_RATE,
        reg=REG,
        m=WIDTH,
        J=GD_STEPS,
       
        seed=42,
    ):
        self.model_names = model_names
        self.feature_dim = feature_dim
        self.learning_rate = learning_rate
        self.reg = reg
        self.m = m
       
        
        self.M = M
        self.L = L
        self.model_name_to_index = {name: i for i, name in enumerate(model_names)}
        self.rng = np.random.default_rng(seed)

        # 网络结构：输入 -> (L-1 个隐藏层, 宽度 m) -> 输出 1
        num_hidden = max(self.L - 1, 1)
        layer_sizes = [self.feature_dim] + [self.m] * num_hidden + [1]
        self.layer_sizes = layer_sizes

        def _block_diag(W):
            z = np.zeros_like(W)
            top = np.concatenate([W, z], axis=1)
            bottom = np.concatenate([z, W], axis=1)
            return np.concatenate([top, bottom], axis=0)

        def _init_params():
            params = []
            for li, (in_dim, out_dim) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:]), start=1):
                # 初始化满足 NTK 结构：
                # W_l = [[W, 0],[0, W]]，W ~ N(0, 4/m)  (l in [L-1])
                # W_L = [w^T, -w^T]，w ~ N(0, 2/m)
                if li < len(layer_sizes) - 1:
                    if in_dim == out_dim and in_dim % 2 == 0:
                        half = in_dim // 2
                        W = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(half, half)).astype(np.float32)
                        w = _block_diag(W).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(4.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                else:
                    if in_dim % 2 == 0:
                        half = in_dim // 2
                        w_vec = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(half,)).astype(np.float32)
                        w = np.concatenate([w_vec, -w_vec], axis=0).reshape(1, -1).astype(np.float32)
                        if w.shape[1] != in_dim:
                            w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    else:
                        w = self.rng.normal(0.0, np.sqrt(2.0 / self.m), size=(out_dim, in_dim)).astype(np.float32)
                    b = np.zeros(out_dim, dtype=np.float32)
                params.append((w, b))
            return params

        def _param_count(params):
            total = 0
            for w, b in params:
                total += w.size + b.size
            return total

        def _build_net_from_params(params):
            net = LLMNet(input_dim=self.feature_dim, width=self.m, L=self.L)
            linear_layers = [m for m in net.net if isinstance(m, nn.Linear)]
            for (w_np, b_np), layer in zip(params, linear_layers):
                with torch.no_grad():
                    layer.weight.copy_(torch.tensor(w_np, dtype=layer.weight.dtype))
                    layer.bias.copy_(torch.tensor(b_np, dtype=layer.bias.dtype))
            return net

        def _flatten_params(net):
            return torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()

        # 维护每个 LLM 的可更新参数
        self.models = {}
        for model_name in self.model_names:
            params = _init_params()
            p = _param_count(params)
            net = _build_net_from_params(params)
            theta0 = _flatten_params(net)
            self.models[model_name] = {
                "params": params,
                "net": net,
                "theta": np.copy(theta0),
                "theta0": np.copy(theta0),
                "Z_diag": np.ones(p, dtype=np.float32) * self.reg,
                "D": [],
                "last_call_time": 0,
            }

    def update_param(self, model_name, feature_vector, reward):
        """
        更新指定 LLM 的参数：
        - 将 (x, v) 记录到数据集 D
        - 使用 D 中所有样本进行梯度下降
        - 目标：最小化 1/2 * sum (f(x;theta) - v)^2 + (m*reg/2)||theta-theta0||^2
        - 更新 Z: Z <- Z + g(x;theta)g(x;theta)^T / m
        """
        model = self.models[model_name]
        model["D"].append((np.array(feature_vector, dtype=np.float32), float(reward)))

        net = model["net"]
        net.train()
        optimizer = optim.Adam(net.parameters(), lr=self.learning_rate)

        xs = torch.tensor([d[0] for d in model["D"]], dtype=torch.float32)
        ys = torch.tensor([d[1] for d in model["D"]], dtype=torch.float32)
        theta0 = torch.tensor(model["theta0"], dtype=torch.float32)

        for _ in range(self.epochs):
            optimizer.zero_grad()
            preds = net(xs).squeeze(-1)
            mse = 0.5 * torch.mean((preds - ys) ** 2)
            theta = torch.cat([p.reshape(-1) for p in net.parameters()])
            reg_term = 0.5 * self.m * self.reg * torch.sum((theta - theta0) ** 2)
            loss = mse + reg_term
            loss.backward()
            optimizer.step()

        # 更新 theta
        model["theta"] = torch.cat([p.detach().reshape(-1) for p in net.parameters()]).cpu().numpy()

        # 更新 Z_diag (用当前 x 的梯度 g)
        x = torch.tensor(feature_vector, dtype=torch.float32)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        net.zero_grad(set_to_none=True)
        y = net(x).sum()
        grads = torch.autograd.grad(y, net.parameters(), retain_graph=False, create_graph=False)
        g = torch.cat([g.reshape(-1) for g in grads]).detach().cpu().numpy()
        model["Z_diag"] += (g ** 2) / float(self.m)

    

    def select_model(self, feature_vector):
        """
        
        """
        
    def respect_rate_limit(self, model_name):
        """Wait if necessary to respect a model's RPM if defined in config."""
        # For the specified OpenRouter models, 'rpm' is not defined, so this block will be skipped.
        model_cfg = MODELS_CONFIG.get(model_name)
        if model_cfg and "rpm" in model_cfg:
            model_state = self.models[model_name]
            min_seconds_between_calls = 60.0 / model_cfg["rpm"]
            time_since_last_call = time.time() - model_state['last_call_time']
            if time_since_last_call < min_seconds_between_calls:
                time.sleep(min_seconds_between_calls - time_since_last_call)

    def save_model_state(self, file_path):
        """Save the model state to a compressed numpy file."""
        state = {
            "config": {
                "feature_dim": self.feature_dim,
                "learning_rate": self.learning_rate,
                "reg": self.reg,
                "m": self.m,
                "epochs": self.epochs,
                "mu": self.mu,
                "M": self.M,
                "L": self.L,
            },
            "models": {},
        }
        for model_name, model_state in self.models.items():
            state["models"][model_name] = {
                "net": model_state["net"].state_dict(),
                "theta": model_state["theta"],
                "theta0": model_state["theta0"],
                "Z_diag": model_state["Z_diag"],
                "last_call_time": model_state["last_call_time"],
            }
        torch.save(state, file_path)

    def load_model_state(self, file_path):
        """Load the model state from a file."""
        state = torch.load(file_path, map_location="cpu", weights_only=False)
        models_state = state.get("models", {})
        for model_name, model_state in models_state.items():
            if model_name not in self.models:
                continue
            net = self.models[model_name]["net"]
            net.load_state_dict(model_state["net"])
            self.models[model_name]["theta"] = np.array(model_state["theta"], dtype=np.float32)
            self.models[model_name]["theta0"] = np.array(model_state["theta0"], dtype=np.float32)
            self.models[model_name]["Z_diag"] = np.array(model_state["Z_diag"], dtype=np.float32)
            self.models[model_name]["last_call_time"] = model_state.get("last_call_time", 0)
        return True